# Notebook for topic modeling 

# 0. Imports

In [13]:
## load packages 
import pandas as pd
import re
import numpy as np

## nltk imports
#!pip install nltk # can install on terminal or by uncommenting this line
import nltk; nltk.download('punkt'); nltk.download('stopwords')
from nltk.tokenize import word_tokenize, wordpunct_tokenize
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

## sklearn imports
from sklearn.feature_extraction.text import CountVectorizer

## lda
#!pip install gensim # can install by uncommenting this line
from gensim import corpora
import gensim

## visualizing LDA--likely need to install
#!pip install pyLDAvis # can install by uncommenting this line
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis
pyLDAvis.enable_notebook()

## print mult things
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## random
import random
import string; punctlist = [char for char in string.punctuation] # list of english punctuation marks

[nltk_data] Downloading package punkt to /Users/wyx/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

[nltk_data] Downloading package stopwords to /Users/wyx/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# 0. Load data

In [14]:
ab = pd.read_csv("../public_data/airbnb_text.zip")
ab.head()

,id,name,name_upper,neighbourhood_group,price
0,2539,Clean & quiet apt home by the park,CLEAN & QUIET APT HOME BY THE PARK,Brooklyn,149
1,2595,Skylit Midtown Castle,SKYLIT MIDTOWN CASTLE,Manhattan,225
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,THE VILLAGE OF HARLEM....NEW YORK !,Manhattan,150
3,3831,Cozy Entire Floor of Brownstone,COZY ENTIRE FLOOR OF BROWNSTONE,Brooklyn,89
4,5022,Entire Apt: Spacious Studio/Loft by central park,ENTIRE APT: SPACIOUS STUDIO/LOFT BY CENTRAL PARK,Manhattan,80


# 1. Preprocess documents

In this case, each name/name_upper, or listing title, we're treating as a document

## 1.1 Load stopwords list and augment with our own custom ones

In [15]:
list_stopwords = stopwords.words("english")

custom_words_toadd = ['apartment', 'new york', 'nyc',
                      'bronx', 'brooklyn',
                     'manhattan', 'queens', 
                      'staten island']

list_stopwords_new = list_stopwords + custom_words_toadd


In [17]:
list_stopwords_new

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

## 1.2 Remove stopwords from lowercase version of corpus


In [21]:
ab.name

0                       Clean & quiet apt home by the park
1                                    Skylit Midtown Castle
2                      THE VILLAGE OF HARLEM....NEW YORK !
3                          Cozy Entire Floor of Brownstone
4         Entire Apt: Spacious Studio/Loft by central park
                               ...                        
48890      Charming one bedroom - newly renovated rowhouse
48891        Affordable room in Bushwick/East Williamsburg
48892              Sunny Studio at Historical Neighborhood
48893                 43rd St. Time Square-cozy single bed
48894    Trendy duplex in the very heart of Hell's Kitchen
Name: name, Length: 48895, dtype: object

In [18]:
## convert to lowercase and a list
corpus_lower = ab.name.str.lower().to_list()
corpus_lower[0:5]

## use wordpunct tokenize and filter out with one
example_listing = corpus_lower[3]
nostop_listing = [word for word in wordpunct_tokenize(example_listing) 
                          if word not in list_stopwords_new]
nostop_listing

['clean & quiet apt home by the park',
 'skylit midtown castle',
 'the village of harlem....new york !',
 'cozy entire floor of brownstone',
 'entire apt: spacious studio/loft by central park']

['cozy', 'entire', 'floor', 'brownstone']

## 1.3 stem and remove non-alpha

Other contexts we may want to leave digits in

In [19]:
## initialize stemmer
porter = PorterStemmer()

## apply to one by iterating
## over the tokens in the list
example_listing_preprocess = [porter.stem(token) 
                            for token in nostop_listing 
                            if token.isalpha() and 
                            len(token) > 2]

example_listing_preprocess

['cozi', 'entir', 'floor', 'brownston']

In [20]:
example_listing
example_listing_preprocess

'cozy entire floor of brownstone'

['cozi', 'entir', 'floor', 'brownston']

## 1.4 Activity 1

The above example performed preprocessing on a single Airbnb listing. We want to generalize this preprocessing across all listings.

- Embed step two (remove stopwords) and step three (stem) into one or two functions that take in a raw string (eg the raw text of an Airbnb review) and return a preprocessed string 
- Apply the function iteratively to preprocess all the texts in `corpus_lower`. Output could either be a list where each list element is a string of a list (e.g., `cozy brownstone apt`), or a list of lists where each element is a tokenized string (e.g., `['cozy', 'brownstone', 'apt'])`

Output is flexible: it could be a list of lists containing tokenized/stemmed text or a list of strings.

In [85]:
# your code here to define the function(s)
def cleantext(raw_text_lower):
    # remove stopwords.
    # raw_text_lower = raw_text.lower().to_list()
    nostopls = [one_tok for one_tok in wordpunct_tokenize(raw_text_lower) 
                          if one_tok not in list_stopwords_new]
    # stem.
    stemmed = [porter.stem(one_tok) 
                            for one_tok in nostopls 
                            if one_tok.isalpha() and 
                            len(one_tok) > 2]
    return stemmed

In [86]:
cleantext(corpus_lower[3])

AttributeError: 'list' object has no attribute 'values'

In [52]:
# A> your code here to apply the function

# RE: corpus_lower = ab.name.str.lower().to_list()
# TypeError: expected string or bytes-like object, got 'float' ----- means there are missing data!!

noNAnames = ab.name[ ~ab.name.isna() ]
corpus_lower_noNA = noNAnames.str.lower().to_list()
# corpus_lower_noNA

cleaned = [cleantext(entry) for entry in corpus_lower_noNA]

['clean & quiet apt home by the park',
 'skylit midtown castle',
 'the village of harlem....new york !',
 'cozy entire floor of brownstone',
 'entire apt: spacious studio/loft by central park',
 'large cozy 1 br apartment in midtown east',
 'blissartsspace!',
 "large furnished room near b'way ",
 'cozy clean guest room - family apt',
 'cute & cozy lower east side 1 bdrm',
 'beautiful 1br on upper west side',
 'central manhattan/near broadway',
 'lovely room 1, garden, best area, legal rental',
 'wonderful guest bedroom in manhattan for singles',
 'west village nest - superhost',
 'only 2 stops to manhattan studio',
 'perfect for your parents + garden',
 'chelsea perfect',
 'hip historic brownstone apartment with backyard',
 'huge 2 br upper east  cental park',
 'sweet and spacious brooklyn loft',
 'cbg ctybgd helpshaiti rm#1:1-4',
 'cbg helps haiti room#2.5',
 'cbg helps haiti rm #2',
 'maison des sirenes1,bohemian apartment',
 'sunny bedroom across prospect park',
 'magnifique suite a

In [65]:
ab_noNA = ab[~ab.name.isna()]

In [66]:
# B. much faster to use apply.
ab_noNA["processed_text"] = ab_noNA.name.apply(cleantext)

/var/folders/9c/c84wrldd0wgbdtr9v0n9gp100000gn/T/ipykernel_50352/4140135987.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ab_noNA["processed_text"] = ab_noNA.name.apply(cleantext)


# 2. Create a document-term matrix and do some basic diagnostics (more manual approach)

Here we'll create a DTM first using the raw documents; in the activity, you'll create one using the preprocessed docs
that you created in activity 1

## 2.1 Define the dtm function and select data to transform into a document-term matrix

In [57]:
## function provided
def create_dtm(list_of_strings, metadata):
    """ 
    Function to create dense document-term matrix (DTM) from a list of strings and provided metadata. 
    A sparse DTM is a list of term_index/doc_index tuples: if a given term occurs in a given doc at least once, 
        then this count is listed as a tuple; if not, that term/doc pair is omitted. 
    In a dense DTM, each row is one text (e.g., an Airbnb listing), each column is a term, and 
        each cell indicates the frequency of that word in that text. 
    
    Parameters:
        list_of_strings (Series): each row contains a preprocessed string (need not be tokenized)
        metadata (DataFrame): contains document-level covariates
    
    Returns:
        Dense DTM with metadata on left and then one column per word in lexicon
    """
    
    # initialize a sklearn tokenizer; this helps us tokenize the preprocessed string input
    vectorizer = CountVectorizer(lowercase = True) 
    dtm_sparse = vectorizer.fit_transform(list_of_strings)
    print('Sparse matrix form:\n', dtm_sparse[:3]) # take a look at sparse representation
    print()
    
    # switch the dataframe from the sparse representation to the normal dense representation (so we can treat it as regular dataframe)
    dtm_dense_named = pd.DataFrame(dtm_sparse.todense(), columns=vectorizer.get_feature_names_out ())
    print('Dense matrix form:\n', dtm_dense_named.head()) # take a look at dense representation
    dtm_dense_named_withid = pd.concat([metadata.reset_index(), dtm_dense_named], axis = 1) # add back document-level covariates

    return(dtm_dense_named_withid)

In [58]:
## filter out na's
## for shorter runtime, random sampling of 1000
## get metadata for those
## and also renaming price col since it's likely to be corpus word
ab_small = ab.loc[~ab.name.isnull(),
           ['id', 'neighbourhood_group', 'price', 'name']].copy().rename(columns = {'price':
            'price_rawdata'}).sample(n = 1000, random_state = 422)

ab_small['name_lower'] = ab_small['name'].str.lower()
ab_small.head()

,id,neighbourhood_group,price_rawdata,name,name_lower
23821,19227560,Queens,100,Super Cozy!,super cozy!
22905,18560625,Brooklyn,30,Beautiful Private Bedroom by Prospect Park,beautiful private bedroom by prospect park
20426,16289576,Manhattan,80,Best Location on the Upper West Side! - Part II,best location on the upper west side! - part ii
2018,893413,Manhattan,2500,Architecturally Stunning Former Synagogue!,architecturally stunning former synagogue!
18790,14882137,Queens,50,"Large, beautiful room near Bushwick","large, beautiful room near bushwick"


## 2.2 Execute the dtm function to create the document-term matrix

In [59]:
## example application on raw lowercase texts; 
dtm_nopre = create_dtm(list_of_strings= ab_small.name_lower,
                      metadata = ab_small[['id', 'neighbourhood_group', 'price_rawdata']])

Sparse matrix form:
 <Compressed Sparse Row sparse matrix of dtype 'int64'
	with 17 stored elements and shape (3, 970)>
  Coords	Values
  (0, 841)	1
  (0, 281)	1
  (1, 152)	1
  (1, 693)	1
  (1, 157)	1
  (1, 205)	1
  (1, 698)	1
  (1, 653)	1
  (2, 165)	1
  (2, 537)	1
  (2, 637)	1
  (2, 856)	1
  (2, 902)	1
  (2, 939)	1
  (2, 774)	1
  (2, 657)	1
  (2, 471)	1

Dense matrix form:
    001  10  10m  10min  10mins  1100  12mins  14  15  15min  ...  yoga  york  \
0    0   0    0      0       0     0       0   0   0      0  ...     0     0   
1    0   0    0      0       0     0       0   0   0      0  ...     0     0   
2    0   0    0      0       0     0       0   0   0      0  ...     0     0   
3    0   0    0      0       0     0       0   0   0      0  ...     0     0   
4    0   0    0      0       0     0       0   0   0      0  ...     0     0   

   you  your  yu  zen  ღღღsteps  法拉盛中心私人房間獨立衛浴  溫馨大套房  獨一無二的紐約閣樓  
0    0     0   0    0         0              0      0          0  
1    0 

In [60]:
## show first set of rows/cols
dtm_nopre.head()

## show arbitrary later cols in resulting data
dtm_nopre.shape
dtm_nopre.iloc[0:5, 480:500]

,index,id,neighbourhood_group,price_rawdata,001,10,10m,10min,10mins,1100,...,yoga,york,you,your,yu,zen,ღღღsteps,法拉盛中心私人房間獨立衛浴,溫馨大套房,獨一無二的紐約閣樓
0,23821,19227560,Queens,100,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,22905,18560625,Brooklyn,30,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,20426,16289576,Manhattan,80,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2018,893413,Manhattan,2500,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,18790,14882137,Queens,50,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


(1000, 974)

,inclusive,incredible,incredibly,indoor,inn,inq,insane,int,interior,international,interns,invincible,inviting,inwood,island,it,italy,its,jefferson,jewel
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 2.3 Use that matrix/column sums to get basic summary stats of top words

In [61]:
## summing each col
top_terms = dtm_nopre[dtm_nopre.columns[4:]].sum(axis = 0)

## sorting from most frequent to least frequent
top_terms.sort_values(ascending = False)

in           367
room         244
private      163
bedroom      152
apartment    130
            ... 
gay            1
gente          1
geodesic       1
george         1
獨一無二的紐約閣樓      1
Length: 970, dtype: int64

## 2.4 Activity 2: repeat the above but using the preprocessed text data

- Stick with the same random sample of 1000 `ab_small`
- Apply the preprocessing steps from activity 1 to create a new column in `ab_small` with the preprocessed text (if you got stuck on that, try just removing stopwords)
- Use the `create_dtm` function to create a document-term matrix from the preprocessed data
- Use colsums to summarize

In [90]:
def cleantext_tosentence(raw_text_lower):
    # remove stopwords.
    clean_sentence = ""
    nostopls = [one_tok for one_tok in wordpunct_tokenize(raw_text_lower) 
                          if one_tok not in list_stopwords_new]
    # stem.
    stemmed = [porter.stem(one_tok) 
                            for one_tok in nostopls 
                            if one_tok.isalpha() and 
                            len(one_tok) > 2]
    
    return " ".join(stemmed) # !!!!!!


In [92]:
## no NA & to string.
# ab_small = ab.loc[~ab.name.isnull(),
#            ['id', 'neighbourhood_group', 'price', 'name']].copy().rename(columns = {'price':
#             'price_rawdata'}).sample(n = 1000, random_state = 422)

# ab_small['name_lower'] = ab_small['name'].str.lower()


ab_small["processed_text"] = ab_small.name_lower.apply(cleantext_tosentence)
ab_small.head()


### OR just use original function and process the column!!
# apply.(" ".join)

,id,neighbourhood_group,price_rawdata,name,name_lower,processed_text
23821,19227560,Queens,100,Super Cozy!,super cozy!,super cozi
22905,18560625,Brooklyn,30,Beautiful Private Bedroom by Prospect Park,beautiful private bedroom by prospect park,beauti privat bedroom prospect park
20426,16289576,Manhattan,80,Best Location on the Upper West Side! - Part II,best location on the upper west side! - part ii,best locat upper west side part
2018,893413,Manhattan,2500,Architecturally Stunning Former Synagogue!,architecturally stunning former synagogue!,architectur stun former synagogu
18790,14882137,Queens,50,"Large, beautiful room near Bushwick","large, beautiful room near bushwick",larg beauti room near bushwick


In [93]:
dtm_pre = create_dtm(list_of_strings= ab_small.processed_text,
                      metadata = ab_small[['id', 'neighbourhood_group', 'price_rawdata']])

Sparse matrix form:
 <Compressed Sparse Row sparse matrix of dtype 'int64'
	with 13 stored elements and shape (3, 753)>
  Coords	Values
  (0, 646)	1
  (0, 170)	1
  (1, 60)	1
  (1, 516)	1
  (1, 65)	1
  (1, 519)	1
  (1, 480)	1
  (2, 70)	1
  (2, 388)	1
  (2, 697)	1
  (2, 729)	1
  (2, 584)	1
  (2, 482)	1

Dense matrix form:
    abcd  abod  access  acidot  acogedor  across  address  ador  aesthet  \
0     0     0       0       0         0       0        0     0        0   
1     0     0       0       0         0       0        0     0        0   
2     0     0       0       0         0       0        0     0        0   
3     0     0       0       0         0       0        0     0        0   
4     0     0       0       0         0       0        0     0        0   

   afford  ...  yard  year  yellow  yoga  york  zen  ღღღstep  法拉盛中心私人房間獨立衛浴  \
0       0  ...     0     0       0     0     0    0        0              0   
1       0  ...     0     0       0     0     0    0        0        

# 3. Use gensim to more automatically preprocess/estimate a topic model

## 3.1 Creating the objects to feed the LDA modeling function

Different outputs described below: 
- Tokenized and preprocessed text 
- Dictionary 
- Corpus 

In [79]:
## Step 1: re-tokenize and store in list
## here, i'm doing with the raw random sample of text
## in activity, you should do with the preprocessed texts
text_raw_tokens = [wordpunct_tokenize(one_text) for one_text in 
                  ab_small.name_lower]


## Step 2: use gensim create dictionary - gets all unique words across documents
text_raw_dict = corpora.Dictionary(text_raw_tokens)
raw_len = len(text_raw_dict) # get length for comparison below

### explore first few keys and values
### see that key is just an arbitrary counter; value is the word itself
{k: text_raw_dict[k] for k in list(text_raw_dict)[:5]}


## Step 3: filter out very rare and very common words
## here, i'm using the threshold that a word needs to appear in at least
## 5% of docs but not more than 95%
## this is an integer count of docs so i round
lower_bound = round(ab_small.shape[0]*0.05)
upper_bound = round(ab_small.shape[0]*0.95)

### apply filtering to dictionary
text_raw_dict.filter_extremes(no_below = lower_bound,
                             no_above = upper_bound)
print(f'Filtering out very rare and very common words reduced the \
length of dictionary from {str(raw_len)} to {str(len(text_raw_dict))}.')
{k: text_raw_dict[k] for k in list(text_raw_dict)[:5]} # show first five entries after filtering


## Step 4: apply dictionary to TOKENIZED texts
## this creates a mapping between each word 
## in a specific listing and the key in the dictionary.
## for words that remain in the filtered dictionary,
## output is a list where len(list) == n documents
## and each element in the list is a list of tuples
## containing the mappings
corpus_fromdict = [text_raw_dict.doc2bow(one_text) 
                   for one_text in text_raw_tokens]

### can apply doc2bow(one_text, return_missing = True) to print words
### eliminated from the listing bc they're not in filtered dictionary.
### but feeding that one with missing values to
### the lda function can cause errors
corpus_fromdict_showmiss = [text_raw_dict.doc2bow(one_text, return_missing = True)
                            for one_text in text_raw_tokens]
print('Sample of documents represented in dictionary format (with omitted words noted):')
corpus_fromdict_showmiss[:10]

{0: '!', 1: 'cozy', 2: 'super', 3: 'beautiful', 4: 'bedroom'}

Filtering out very rare and very common words reduced the length of dictionary from 1047 to 31.


{0: '!', 1: 'cozy', 2: 'beautiful', 3: 'bedroom', 4: 'park'}

Sample of documents represented in dictionary format (with omitted words noted):


[([(0, 1), (1, 1)], {'super': 1}),
 ([(2, 1), (3, 1), (4, 1), (5, 1)], {'by': 1, 'prospect': 1}),
 ([(0, 1), (6, 1), (7, 1)],
  {'best': 1,
   'ii': 1,
   'location': 1,
   'on': 1,
   'part': 1,
   'side': 1,
   'upper': 1,
   'west': 1}),
 ([(0, 1)],
  {'architecturally': 1, 'former': 1, 'stunning': 1, 'synagogue': 1}),
 ([(2, 1), (8, 1), (9, 1), (10, 1), (11, 1)], {'bushwick': 1}),
 ([(4, 1), (8, 1), (9, 1), (12, 1), (13, 2)],
  {'bath': 1, 'bed': 1, 'by': 1, 'central': 1, 'college': 1, 'hunter': 1}),
 ([(9, 1), (11, 1), (14, 1), (15, 1)], {'bohemian': 1, 'brownstone': 1}),
 ([(16, 1)],
  {'fidi': 1, 'huge': 1, 'loft': 1, 'views': 1, 'w': 1, 'water': 1}),
 ([], {'hillside': 1, 'hotel': 1}),
 ([(5, 1), (9, 1), (11, 1), (14, 1), (15, 1)], {'airy': 1})]

##  3.2 Estimating the model

In [80]:
## Step 5: we're finally ready to estimate the model!
## full documentation here - https://radimrehurek.com/gensim/models/ldamodel.html
## here, we're feeding the lda function:
## (1) the corpus we created from the dictionary,
## (2) a parameter we decide on for the number of topics (k),
## (3) the dictionary itself,
## (4) parameter for number of passes through training data (more means slower), and
## (5) parameter that returns, for each word remaining in dict, the topic probabilities.
## see documentation for many other arguments you can vary
ldamod = gensim.models.ldamodel.LdaModel(corpus_fromdict, 
                                         num_topics = 5, 
                                         id2word=text_raw_dict, 
                                         passes=6, 
                                         alpha = 'auto',
                                         per_word_topics = True)

print(type(ldamod))



<class 'gensim.models.ldamodel.LdaModel'>


## 3.3  Seeing what topics the estimated model discovers

In [81]:
## Post-model 1: explore corpus-wide summary of topics
### getting the topics and top words; can retrieve diff top words
topics = ldamod.print_topics(num_words = 10)
for topic in topics:
    print(topic)


(0, '0.173*"bedroom" + 0.120*"brooklyn" + 0.101*"in" + 0.088*"beautiful" + 0.078*"!" + 0.076*"apartment" + 0.048*"1" + 0.040*"spacious" + 0.038*"private" + 0.034*"apt"')
(1, '0.148*"in" + 0.142*"studio" + 0.130*"2" + 0.115*"apartment" + 0.086*"the" + 0.058*"of" + 0.053*"large" + 0.036*"-" + 0.034*"bedroom" + 0.032*"east"')
(2, '0.206*"room" + 0.194*"in" + 0.135*"private" + 0.060*"&" + 0.058*"williamsburg" + 0.046*"/" + 0.042*"cozy" + 0.041*"," + 0.038*"spacious" + 0.024*"with"')
(3, '0.167*"," + 0.130*"to" + 0.104*"sunny" + 0.089*"park" + 0.056*"room" + 0.051*"cozy" + 0.047*"manhattan" + 0.044*"bedroom" + 0.044*"near" + 0.043*"-"')
(4, '0.135*"-" + 0.117*"," + 0.106*"!" + 0.084*"1" + 0.076*"apt" + 0.057*"." + 0.053*"in" + 0.052*"and" + 0.052*"east" + 0.047*"/"')


In [82]:
## Post-model 2: explore topics associated with each document
### for each item in our original dictionary, get list of topic probabilities
l=[ldamod.get_document_topics(item) for item in corpus_fromdict]
### print result
text_raw_tokens[0:5]
l[0:5]

[['super', 'cozy', '!'],
 ['beautiful', 'private', 'bedroom', 'by', 'prospect', 'park'],
 ['best',
  'location',
  'on',
  'the',
  'upper',
  'west',
  'side',
  '!',
  '-',
  'part',
  'ii'],
 ['architecturally', 'stunning', 'former', 'synagogue', '!'],
 ['large', ',', 'beautiful', 'room', 'near', 'bushwick']]

[[(0, np.float32(0.053846054)),
  (1, np.float32(0.054958135)),
  (2, np.float32(0.06733787)),
  (3, np.float32(0.05194565)),
  (4, np.float32(0.7719123))],
 [(0, np.float32(0.8673497)),
  (1, np.float32(0.03187286)),
  (2, np.float32(0.03900259)),
  (3, np.float32(0.030494127)),
  (4, np.float32(0.0312806))],
 [(0, np.float32(0.039624266)),
  (1, np.float32(0.28869197)),
  (2, np.float32(0.04861639)),
  (3, np.float32(0.037977852)),
  (4, np.float32(0.5850895))],
 [(0, np.float32(0.08441116)),
  (1, np.float32(0.085620776)),
  (2, np.float32(0.10330539)),
  (3, np.float32(0.08069114)),
  (4, np.float32(0.64597154))],
 [(0, np.float32(0.27636862)),
  (1, np.float32(0.026478639)),
  (2, np.float32(0.03250183)),
  (3, np.float32(0.6388118)),
  (4, np.float32(0.025839083))]]

### Visualizing 

In [2]:
lda_display = gensimvis.prepare(ldamod, corpus_fromdict, text_raw_dict)
pyLDAvis.display(lda_display)
# pprint(lda_model.print_topics())

NameError: name 'gensimvis' is not defined

## 3.4 Activity 3

- Preprocess the texts if you haven't already
- Run the topic model with preprocessed texts
- Play around with other parameters like `n_topics` to find a configuration that produces useful topics

If you get stuck on the preprocessing part, you can use below function and example code for applying it. Then continue as above (start with tokenizing).

In [3]:
# PROCESSED: ab_small["processed_text"] & dtm_pre
# 其实不用改varaible名字。改内容就行，不重要。


## Step 1: re-tokenize and store in list
## here, i'm doing with the raw random sample of text
## in activity, you should do with the preprocessed texts
text_raw_tokens1 = [wordpunct_tokenize(one_text) for one_text in 
                  ab_small.processed_text]

## Step 2: use gensim create dictionary - gets all unique words across documents
text_raw_dict1 = corpora.Dictionary(text_raw_tokens1)
raw_len1 = len(text_raw_dict1) # get length for comparison below

### explore first few keys and values
### see that key is just an arbitrary counter; value is the word itself
{k: text_raw_dict1[k] for k in list(text_raw_dict1)[:5]}


## Step 3: filter out very rare and very common words
## here, i'm using the threshold that a word needs to appear in at least
## 5% of docs but not more than 95%
## this is an integer count of docs so i round
lower_bound1 = round(ab_small.shape[0]*0.05)
upper_bound1 = round(ab_small.shape[0]*0.95)

### apply filtering to dictionary
text_raw_dict1.filter_extremes(no_below = lower_bound1,
                             no_above = upper_bound1)
print(f'Filtering out very rare and very common words reduced the \
length of dictionary from {str(raw_len1)} to {str(len(text_raw_dict1))}.')
{k: text_raw_dict1[k] for k in list(text_raw_dict1)[:5]} # show first five entries after filtering


## Step 4: apply dictionary to TOKENIZED texts
## this creates a mapping between each word 
## in a specific listing and the key in the dictionary.
## for words that remain in the filtered dictionary,
## output is a list where len(list) == n documents
## and each element in the list is a list of tuples
## containing the mappings
corpus_fromdict1 = [text_raw_dict1.doc2bow(one_text) 
                   for one_text in text_raw_tokens1]

### can apply doc2bow(one_text, return_missing = True) to print words
### eliminated from the listing bc they're not in filtered dictionary.
### but feeding that one with missing values to
### the lda function can cause errors
corpus_fromdict_showmiss1 = [text_raw_dict1.doc2bow(one_text, return_missing = True)
                            for one_text in text_raw_tokens1]
print('Sample of documents represented in dictionary format (with omitted words noted):')
corpus_fromdict_showmiss1[:10]

NameError: name 'ab_small' is not defined

In [105]:
## Step 5: we're finally ready to estimate the model!
## full documentation here - https://radimrehurek.com/gensim/models/ldamodel.html
## here, we're feeding the lda function:
## (1) the corpus we created from the dictionary,
## (2) a parameter we decide on for the number of topics (k),
## (3) the dictionary itself,
## (4) parameter for number of passes through training data (more means slower), and
## (5) parameter that returns, for each word remaining in dict, the topic probabilities.
## see documentation for many other arguments you can vary
ldamod = gensim.models.ldamodel.LdaModel(corpus_fromdict1, 
                                         num_topics = 5, 
                                         id2word=text_raw_dict1, 
                                         passes=6, 
                                         alpha = 'auto',
                                         per_word_topics = True)

print(type(ldamod))



<class 'gensim.models.ldamodel.LdaModel'>


In [106]:
## Post-model 1: explore corpus-wide summary of topics
### getting the topics and top words; can retrieve diff top words
topics = ldamod.print_topics(num_words = 10)
for topic in topics:
    print(topic)

(0, '0.359*"privat" + 0.236*"room" + 0.188*"studio" + 0.127*"larg" + 0.052*"near" + 0.023*"beauti" + 0.004*"apt" + 0.003*"bedroom" + 0.003*"spaciou" + 0.002*"east"')
(1, '0.376*"spaciou" + 0.217*"east" + 0.171*"sunni" + 0.086*"apt" + 0.075*"studio" + 0.020*"beauti" + 0.014*"room" + 0.012*"bedroom" + 0.011*"privat" + 0.007*"larg"')
(2, '0.315*"park" + 0.305*"apt" + 0.191*"williamsburg" + 0.064*"beauti" + 0.053*"larg" + 0.039*"near" + 0.011*"bedroom" + 0.006*"studio" + 0.006*"room" + 0.003*"east"')
(3, '0.589*"room" + 0.138*"cozi" + 0.104*"near" + 0.085*"williamsburg" + 0.048*"sunni" + 0.010*"larg" + 0.008*"bedroom" + 0.005*"apt" + 0.004*"studio" + 0.002*"park"')
(4, '0.499*"bedroom" + 0.198*"cozi" + 0.110*"beauti" + 0.109*"privat" + 0.053*"sunni" + 0.010*"larg" + 0.006*"room" + 0.005*"williamsburg" + 0.004*"park" + 0.002*"near"')


In [107]:
## Post-model 2: explore topics associated with each document
### for each item in our original dictionary, get list of topic probabilities
l=[ldamod.get_document_topics(item) for item in corpus_fromdict1]
### print result
text_raw_tokens1[0:5]
l[0:5]

[['super', 'cozi'],
 ['beauti', 'privat', 'bedroom', 'prospect', 'park'],
 ['best', 'locat', 'upper', 'west', 'side', 'part'],
 ['architectur', 'stun', 'former', 'synagogu'],
 ['larg', 'beauti', 'room', 'near', 'bushwick']]

[[(0, np.float32(0.10973305)),
  (1, np.float32(0.09398391)),
  (2, np.float32(0.09017684)),
  (3, np.float32(0.0997194)),
  (4, np.float32(0.6063868))],
 [(0, np.float32(0.046677794)),
  (1, np.float32(0.037557367)),
  (2, np.float32(0.26492435)),
  (3, np.float32(0.038893018)),
  (4, np.float32(0.6119475))],
 [(0, np.float32(0.22041161)),
  (1, np.float32(0.18874101)),
  (2, np.float32(0.1811334)),
  (3, np.float32(0.19576207)),
  (4, np.float32(0.21395183))],
 [(0, np.float32(0.22041161)),
  (1, np.float32(0.18874101)),
  (2, np.float32(0.1811334)),
  (3, np.float32(0.19576207)),
  (4, np.float32(0.21395183))],
 [(0, np.float32(0.84054697)),
  (1, np.float32(0.037664175)),
  (2, np.float32(0.036644068)),
  (3, np.float32(0.04019696)),
  (4, np.float32(0.04494779))]]

In [109]:
lda_display = gensimvis.prepare(ldamod, corpus_fromdict1, text_raw_dict1)
pyLDAvis.display(lda_display)